In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from catboost import CatBoostRegressor

In [52]:
train_data = pd.read_csv("dataset/train.csv")
test_data = pd.read_csv("dataset/test.csv")

In [53]:
X = train_data.drop(columns = ['SalePrice', 'Id', 'GarageArea', 'TotRmsAbvGrd'])
y = train_data['SalePrice']
X_test = test_data.drop(columns = ['Id', 'GarageArea', 'TotRmsAbvGrd'])

In [54]:
X_categorical = X.select_dtypes(include = ['object']).columns.tolist()
X_numerical = X.select_dtypes(exclude = ['object']).columns.tolist()

In [55]:
numerical_transformer = SimpleImputer(strategy = 'mean')

In [56]:
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

In [57]:
preprocessor = ColumnTransformer(transformers = [
    ('cat', categorical_transformer, X_categorical),
    ('num', numerical_transformer, X_numerical)
])

In [58]:
model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('catboost', CatBoostRegressor(verbose=0, random_state=42))
])

In [59]:
param_grid = {
    'catboost__iterations': [100, 200, 300],
    'catboost__learning_rate': [0.05, 0.1, 0.2],
    'catboost__depth': [3, 4, 5]
}


In [60]:
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error')

In [61]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['MSZoning',
                                                                          'Street',
                                                                          'Alley',
                                                                          'LotShape',
                                                                          'LandContour',
                                                                          'Utilities',
                                                                          'LotConfig',
                                                                          'LandSlope',
                                                                          'Neighborhood',
                                                                          'Condition1',
                                                                          'Condition2',...
                                                                          'KitchenAbvGr',
                                                                          'Fireplaces',
                                                                          'GarageYrBlt',
                                                                          'GarageCars',
                                                                          'WoodDeckSF',
                                                                          'OpenPorchSF',
                                                                          'EnclosedPorch',
                                                                          '3SsnPorch',
                                                                          'ScreenPorch', ...])])),
                                       ('catboost',
                                        <catboost.core.CatBoostRegressor object at 0x000001F945485880>)]),
             param_grid={'catboost__depth': [3, 4, 5],
                         'catboost__iterations': [100, 200, 300],
                         'catboost__learning_rate': [0.05, 0.1, 0.2]},
             scoring='neg_mean_squared_error')

In [63]:
y_pred = grid_search.predict(X_val)

In [64]:
y_test = grid_search.predict(X_test)

In [65]:
output = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': y_test})
output.to_csv('TreeEnsembles_catboost.csv', index=False)